# Benchmarking ASR (Automatic Speech Recognition) Models

## Overview

This notebook benchmarks three state-of-the-art Automatic Speech Recognition (ASR)
models for a hypothetical voice-based AI assistant deployed in customer support call
environments — characterized by noisy audio, multiple accents, and conversational speech.

**Evaluation Criteria:** Word Error Rate (WER), Inference Time, and GPU Memory Usage.  
**Hardware:** Google Colab T4 GPU  
**Dataset:** People's Speech (MLCommons) — dirty test split

## Loading dependencies

In [ ]:
! pip install huggingface transformers torch torchaudio jiwer nemo_toolkit['asr']

In [ ]:
import torch
import transformers
import logging
from transformers import AutoProcessor, AutoModelForCTC
import torchaudio
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import re
from jiwer import wer
import nemo.collections.asr as nemo_asr
import time
import soundfile as sf
import tempfile
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

transformers.logging.set_verbosity_error()
logging.getLogger('nemo_logger').setLevel(logging.ERROR)

## Dataset Selection

The **People's Speech dataset (dirty, test split)** was selected for the following reasons:

- **Noisy & conversational:** The `dirty` configuration contains unfiltered, real-world
  audio with background noise, varied recording quality, and natural speech patterns
  (pauses, stammers), closely mirroring actual customer support call conditions.
- **No data leakage:** The `test` split was specifically chosen to avoid overlap with
  training data used by any of the evaluated models, ensuring unbiased benchmarking.
- **NVIDIA validation:** People's Speech was used in training NVIDIA's Parakeet and
  Canary models, further validating its relevance as a benchmark dataset.
- **Streaming mode:** Due to the dataset's large size (~1-2TB), streaming mode with
  `take(100)` was used to efficiently sample 100 test examples without full download.

**Alternatives considered and rejected:**
- *LibriSpeech:* Too clean (audiobook recordings), not representative of noisy calls.
- *Common Voice (Mozilla):* Moved off HuggingFace as of October 2025, unavailable
  via the datasets library.

## Loading the Dataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("MLCommons/peoples_speech", "dirty", data_dir="test", split="test",
                  streaming=True)

dataset = dataset.take(100)

In [ ]:
# To clean the ground truth (Reference Text).
# When calculating WER, any discrepancies will count as errors even if the model transcribes correctly.

def clean_text(text):
  text = re.sub(r'\[.*?\]', '', text)
  text = text.lower().strip()

  return text

## Model Selection

Three models were selected to maximize architectural diversity while remaining
feasible on a Colab T4 GPU:

| Model | Architecture | Key Strength |
|---|---|---|
| **Wav2Vec2 (facebook/wav2vec2-base-960h)** | Self-supervised CNN + CTC | Baseline; fast inference |
| **Whisper Small (openai/whisper-small)** | Encoder-Decoder Transformer | Multilingual; punctuation-aware output |
| **Parakeet TDT 1.1B (nvidia/parakeet-tdt-1.1b)** | Conformer + RNN-Transducer | Real-time optimized; low latency |

**Models considered but excluded:**
- *Whisper Large V3 Turbo:* Superior accuracy but too memory-intensive for Colab T4.
- *Distil-Whisper:* Shares encoder-decoder architecture with Whisper Small,
  insufficient architectural diversity.
- *Parakeet TDT via NeMo:* Successfully integrated despite heavier dependency
  footprint of the NeMo toolkit.
- *Gnani.ai Vachana ASR:* No open-source weights available on HuggingFace.
  Primarily targets CX/voice agent applications in Indian languages (Kannada-focused).
  Strong contender for Indic language deployments but outside scope of this benchmark.

**Note on hardware constraints:** SOTA 2026 models like Whisper Large V3 Turbo and
Canary-Qwen 2.5B were identified during research but require more GPU memory than
available on Colab T4. Lighter variants were selected to ensure reproducibility.

## Loading the Models

In [ ]:
wav2vec2_processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec2_model = AutoModelForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(device)

whisper_processor = AutoProcessor.from_pretrained("openai/whisper-small")
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-small").to(device)

parakeet_model = nemo_asr.models.ASRModel.from_pretrained(
    "nvidia/parakeet-tdt-1.1b"
).to(device)

In [ ]:
wav2vec2_model.eval()
whisper_model.eval()
parakeet_model.eval()

## Model Inference

In [ ]:
def infer_wav2vec2(audio_array, sampling_rate: int=16000):

  start_time = time.time()

  torch.cuda.reset_peak_memory_stats()

  input_values = wav2vec2_processor(audio_array, sampling_rate=sampling_rate,
                                    return_tensors="pt", padding="longest").input_values.to(device)

  with torch.no_grad():
    logits = wav2vec2_model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = wav2vec2_processor.batch_decode(predicted_ids)

  end_time = time.time()

  inference_memory = torch.cuda.max_memory_allocated()

  inference_time = end_time - start_time

  return transcription[0], inference_time, inference_memory

In [ ]:
def infer_whisper(audio_array, sampling_rate: int = 16000):

  start_time = time.time()

  torch.cuda.reset_peak_memory_stats()

  inputs = whisper_processor(audio_array,
                              sampling_rate=sampling_rate,
                              return_tensors="pt",
                              padding="max_length",
                              truncation=True)

  input_features = inputs.input_features.to(device)
  attention_mask = inputs.attention_mask.to(device) if hasattr(inputs, 'attention_mask') else None


  with torch.no_grad():
    predicted_ids = whisper_model.generate(input_features,
                                           attention_mask=attention_mask)
    transcription = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)

  end_time = time.time()

  inference_time = end_time - start_time
  inference_memory = torch.cuda.max_memory_allocated()

  return transcription[0], inference_time, inference_memory

In [ ]:
def infer_parakeet(audio_array, sampling_rate: int = 16000):
  start_time = time.time()

  torch.cuda.reset_peak_memory_stats()

  with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
    sf.write(f.name, audio_array, sampling_rate)
    temp_path = f.name

  transcription = parakeet_model.transcribe([temp_path])
  os.remove(temp_path)

  end_time = time.time()

  inference_time = end_time - start_time
  inference_memory = torch.cuda.max_memory_allocated()

  transcription_text = transcription[0].text if hasattr(transcription[0], 'text') else str(transcription[0])

  return transcription_text, inference_time, inference_memory

## Model Benchmarking

In [ ]:
def benchmark_asr_models():
  results = {
      "wav2vec2": {"wer_error": [], "infer_time": [], "infer_memory": []},
      "whisper": {"wer_error": [], "infer_time": [], "infer_memory": []},
      "parakeet": {"wer_error": [], "infer_time": [], "infer_memory": []}
      }

  for sample in dataset:
    audio_array = sample['audio']['array']
    sampling_rate = sample['audio']['sampling_rate']
    reference_text = clean_text(sample['text'])

    # wav2vec2
    wav2vec2_transc, inf_time, inf_mem = infer_wav2vec2(audio_array, sampling_rate)
    wer_error = wer(reference_text, wav2vec2_transc)

    results["wav2vec2"]["wer_error"].append(wer_error)
    results["wav2vec2"]["infer_time"].append(inf_time)
    results["wav2vec2"]["infer_memory"].append(inf_mem)

    # whisper
    whisper_transc, inf_time, inf_mem = infer_whisper(audio_array, sampling_rate)
    wer_error = wer(reference_text, whisper_transc)

    results["whisper"]["wer_error"].append(wer_error)
    results["whisper"]["infer_time"].append(inf_time)
    results["whisper"]["infer_memory"].append(inf_mem)

    # parakeet
    parakeet_transc, inf_time, inf_mem = infer_parakeet(audio_array, sampling_rate)
    wer_error = wer(reference_text, parakeet_transc)

    results["parakeet"]["wer_error"].append(wer_error)
    results["parakeet"]["infer_time"].append(inf_time)
    results["parakeet"]["infer_memory"].append(inf_mem)

  return results

In [ ]:
def aggregate_results(results):
  agg_results = {
      "wav2vec2": {"mean_wer": None, "mean_infer_time": None, "mean_infer_mem": None},
      "whisper": {"mean_wer": None, "mean_infer_time": None, "mean_infer_mem": None},
      "parakeet": {"mean_wer": None, "mean_infer_time": None, "mean_infer_mem": None}
      }

  for model_name, metrics in results.items():
    agg_results[model_name] = {
        "mean_wer": np.mean(metrics["wer_error"]),
        "mean_infer_time": np.mean(metrics["infer_time"]),
        "mean_infer_mem": np.mean(metrics["infer_memory"])
        }

  return agg_results

In [ ]:
def display_results(agg_results):
  df = pd.DataFrame(agg_results).T.reset_index()
  df.columns = ["Model", "Mean WER", "Mean Inference Time (s)", "Mean Memory Used (MB)"]
  df["Mean WER"] = df["Mean WER"].round(4)
  df["Mean Inference Time (s)"] = df["Mean Inference Time (s)"].round(4)
  df["Mean Memory Used (MB)"] = (df["Mean Memory Used (MB)"] / 1e6).round(2)

  print(df.to_string(index=False))

  return df

In [ ]:
results = benchmark_asr_models()
agg_results = aggregate_results(results)

In [ ]:
df = display_results(agg_results)

In [ ]:
df.to_csv("benchmark_results.csv", index=False)

## Visualizing the Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("ASR Model Benchmark Comparison", fontsize=16, fontweight='bold')

sns.barplot(data=df, x="Model", y="Mean WER", hue="Model", ax=axes[0])
axes[0].set_title("Word Error Rate (lower is better)")
axes[0].set_ylabel("Mean WER")

sns.barplot(df, x="Model", y="Mean Inference Time (s)", hue="Model", ax=axes[1])
axes[1].set_title("Mean Inference Time (lower is better)")
axes[1].set_ylabel("Mean Inference Time (s)")

sns.barplot(df, x="Model", y="Mean Memory Used (MB)", hue="Model", ax=axes[2])
axes[2].set_title("Mean Memory Used (lower is better)")
axes[2].set_ylabel("Mean Memory Used (MB)")

plt.tight_layout()
plt.savefig("benchmark_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## Qualitative Sample

### Qualitative Analysis

Beyond aggregate metrics, a single sample was passed through all three models to
illustrate qualitative differences in transcription behavior.

**Key observations:**
- **Wav2Vec2** produces all-caps output with missing punctuation and word-level errors
  ("ET'S" instead of "That's", "ACTION" instead of "actually"). This is consistent
  with its high WER and reflects CTC decoding without a language model.
- **Whisper Small** produces the most readable output, correct punctuation, proper
  sentence structure, and numbers written as digits ("1960s and 70s"). Notably,
  Whisper's added punctuation *increases* its technical WER relative to the
  unpunctuated reference text, meaning its actual linguistic accuracy is higher
  than the WER metric suggests.
- **Parakeet TDT** achieves the lowest WER by closely matching the reference text
  style, but reads numbers as spoken words ("one nine hundred sixty s and seventies"), a known characteristic of Transducer models optimized for raw speech fidelity
  over written text normalization.

**Implication:** WER alone is insufficient for production evaluation. For customer
support deployments, Parakeet's lower WER must be weighed against Whisper's superior
text normalization and readability, which reduces post-processing requirements.

In [ ]:
sample = next(iter(dataset))

audio_array = sample['audio']['array']
sampling_rate = sample['audio']['sampling_rate']
reference_text = sample['text']

print(f"Sampling Rate: {sampling_rate}")

wav2vec2_transc, _, _ = infer_wav2vec2(audio_array, sampling_rate)
whisper_transc, _, _ = infer_whisper(audio_array, sampling_rate)
parakeet_transc, _, _ = infer_parakeet(audio_array, sampling_rate)

print(f"\n\nReference Text:\n{reference_text}")
print(f"Wav2Vec2 Transcription:\n{wav2vec2_transc}")
print(f"WhisperTranscription:\n{whisper_transc}")
print(f"Parakeet Transcription:\n{parakeet_transc}")